In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#/content/drive/MyDrive/avito/train.parquet

In [ ]:
import pandas as pd

# Для начала просто посмотрим

In [ ]:
df = pd.read_parquet("/content/drive/MyDrive/avito/train.parquet")

query_cols = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

df["query_key"] = (
    df[query_cols]
    .fillna("")
    .astype(str)
    .agg(" || ".join, axis=1)
)

query_stats = (
    df.groupby("query_key")
      .agg(
          n_rows=("item_id", "size"),
          n_items=("item_id", "nunique")
      )
)

print(query_stats.describe())

              n_rows        n_items
count  354463.000000  354463.000000
mean        1.404020       1.317638
std         2.607192       1.611180
min         1.000000       1.000000
25%         1.000000       1.000000
50%         1.000000       1.000000
75%         1.000000       1.000000
max       628.000000     278.000000


In [ ]:
print("rows:", len(df))
print("unique queries:", df["search_query"].nunique())

rows: 497673
unique queries: 74529


# Что же ищут пользователи?

In [ ]:
df["search_query"].value_counts().head(30)

,count
search_query,
маникюр,6540
массаж,5702
наращивание ресниц,5395
установка кондиционеров,4573
педикюр,3436
сантехник,3405
вывоз мусора,3278
эвакуатор,3263
электрик,3158


# Очень много повторяющихся запросов. Следующий вопрос - являются ли локация и рейтинг важным сигналом?

In [ ]:
same_loc = (
    df["search_location_id"]
    ==
    df["item_location_id"]
).mean()

print(same_loc)

0.8310195650557696


In [ ]:
df.groupby(
    pd.qcut(
        df["item_rating"],
        10,
        duplicates="drop"
    )
).size()

/tmp/ipykernel_4021/2754416991.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(


,0
item_rating,
"(-0.001, 4.659]",46945
"(4.659, 4.837]",46951
"(4.837, 4.915]",46971
"(4.915, 4.963]",47316
"(4.963, 5.0]",281258


**Локация - очень важная фича, а вот рейтинг не очень равномерно распределен среди итемов**

In [ ]:
# сколько microcat на запрос

df.groupby("search_query")["item_microcat_id"] \
  .nunique() \
  .describe()


,item_microcat_id
count,74529.000000
mean,1.348965
std,1.025439
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,32.000000


In [ ]:

# сколько category на запрос

df.groupby("search_query")["item_category_id"] \
  .nunique() \
  .describe()



,item_category_id
count,74529.000000
mean,1.000174
std,0.013206
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,2.000000


In [ ]:
# сколько location на запрос

df.groupby("search_query")["item_location_id"] \
  .nunique() \
  .describe()


,item_location_id
count,74529.000000
mean,3.986717
std,15.824029
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,707.000000


In [ ]:

# сколько уникальных item на запрос

df.groupby("search_query")["item_id"] \
  .nunique() \
  .describe()

,item_id
count,74529.000000
mean,5.974332
std,50.302240
min,1.000000
25%,1.000000
50%,1.000000
75%,3.000000
max,5474.000000
